# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. You will load data described by the Croissant schema, examine its entities through their `@id` references, and perform basic analysis and visualization.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each record set, field, and column is referenced by its `@id`. We first discover what record sets are available.

In [ ]:
# List available record set @id's
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets defined explicitly in the Croissant 'recordSet' field.")
    # Some Croissant schemas may define recordSets only implicitly (via schema:distribution), so try to introspect them
else:
    print("Available record sets and their @ids:")
    for rs in record_sets:
        print(f"- {rs['@id']}: {rs.get('name', '')}")

# For this schema, let's use the mlcroissant Dataset API to list discoverable record sets anyhow
print("\nRecord Sets detected by mlcroissant:")
for record_set in dataset.record_sets:
    rs_id = record_set['@id']
    name = record_set.get('name', '(no name)')
    description = record_set.get('description', '')
    print(f"@id: {rs_id}, name: {name}, description: {description}")

# For each record set, list its fields with @id
for record_set in dataset.record_sets:
    print(f"\nFields for Record Set: {record_set['@id']}")
    if 'field' in record_set:
        fields = record_set['field']
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"  Field @id: {field_id}")
    else:
        print("  (No fields listed)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Discover all record set @ids found by mlcroissant
record_set_ids = [r['@id'] for r in dataset.record_sets]
print(f"Record Set @ids: {record_set_ids}")

# Attempt to load data for each record set
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nLoaded record set {record_set_id}: shape {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
        else:
            print(f"No records found in record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Display the head of the main record set (pick the largest one, or the first if only one)
if dataframes:
    main_rs = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"\nPreview of main data in record set '@id': {main_rs}")
    display(dataframes[main_rs].head())
else:
    print("No tabular record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates filtering and grouping for exploration.

Field, record set, and column/field names are referenced by their `@id` where possible.

In [ ]:
# Select the main record set
if dataframes:
    record_set_id = main_rs
    df = dataframes[record_set_id].copy()
    print(f"Working with record set: {record_set_id}, shape: {df.shape}")

    # List numeric fields by checking dtype
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields: {numeric_fields}")

    # If available, pick a representative numeric field (e.g., 'second_primary_age')
    numeric_field = numeric_fields[0] if numeric_fields else None

    if numeric_field is not None:
        print(f"Analyzing numeric field: {numeric_field}")

        # Example threshold: median + 1 std
        threshold = df[numeric_field].median() + df[numeric_field].std()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Records with {numeric_field} > {threshold:.1f}: {len(filtered_df)} out of {len(df)}")
        print(filtered_df[[numeric_field]].head())

        # Normalize the numeric field in the filtered dataset
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field (e.g., 'sex', 'msi_status', or similar)
        candidate_cats = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field]
        group_field = None
        for gfield in ['sex', 'gender', 'msi_status', 'msi_h_status', 'anatomical_location']:
            if gfield in candidate_cats:
                group_field = gfield
                break
        if (group_field is None) and candidate_cats:
            group_field = candidate_cats[0]

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. This example will plot the distribution of a selected numeric field and, if possible, boxplots by group.

In [ ]:
if dataframes and (numeric_field is not None):
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        df.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset described with a Croissant schema using the `mlcroissant` library. We loaded the dataset, listed available record sets and their associated fields by `@id`, loaded records into DataFrames, and performed basic exploratory and visualization steps.

- Make sure to always reference entities (record sets, fields, columns) by their `@id` for reproducibility.
- You can extend this notebook to perform more domain-specific statistical analysis or modeling on the dataset.

For further information about the dataset, including detailed variable definitions, see the official dataset [metadata and documentation](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).
